<a href="https://colab.research.google.com/github/tkach3nkoo/MN/blob/main/%D0%9B%D0%A06_%D0%A2%D0%BA%D0%B0%D1%87%D0%B5%D0%BD%D0%BA%D0%BE_%D0%B3%D1%80%D1%83%D0%BF%D0%B0_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Лабораторна робота № 6 — ШАБЛОН ДЛЯ СТУДЕНТІВ**

**Класифікація. Запобігання перенавчанню**

Датасет: https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset?select=diabetes_prediction_dataset.csv

Виконуйте завдання послідовно. **До аналізу кореляцій і викидів спочатку зробіть train/test split.** В усіх завданнях для дослідження кореляцій, меж IQR і асиметрії використовуйте **тільки train**. Не видаляйте викиди автоматично, не балансуйте test. Заповніть код у клітинках `TODO`, додайте короткі висновки та результати.

Потрібно навчити LogisticRegression, RidgeClassifier, SGDClassifier, SVC, вивести звіти та матриці плутанини, підібрати параметри методом HalvingGridSearchCV (`cv=3`) і показати 10 тестових прогнозів.

*Виберіть лише один зі способів завантаження даних нижче.*

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

**Завантаження датасету.** Спочатку використовується локальний CSV, а якщо його немає — відкритий CSV з GitHub (копія датасету Kaggle). Для Colab можна завантажити CSV у робочу теку вручну.

In [ ]:
from pathlib import Path
csv_path = Path("diabetes_prediction_dataset.csv")
csv_url = "https://raw.githubusercontent.com/mezkymy/diabetes-prediction/main/diabetes_prediction_dataset.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    try:
        df = pd.read_csv(csv_url)
    except Exception as exc:
        raise FileNotFoundError(
            "Додайте diabetes_prediction_dataset.csv до робочої теки або запустіть блокнот з доступом до інтернету."
        ) from exc
print(f"Розмір: {df.shape[0]:,} рядків × {df.shape[1]} стовпців")
display(df.head())

In [ ]:
# Дані вже завантажені у попередній клітинці.
display(df.head())

**Джерело даних:** [Kaggle — Diabetes Prediction Dataset](https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset). Альтернативне завантаження через Kaggle не потрібне.

In [ ]:
# Використовується один спосіб завантаження даних з клітинки вище.

Using Colab cache for faster access to the 'diabetes-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/diabetes-prediction-dataset


In [ ]:
display(df.sample(5, random_state=RANDOM_STATE))

Вивести інформацію про датасет

In [ ]:
df.info()
print("Категоріальні ознаки: gender, smoking_history; інші — числові.")

Провести дослідження категоріальних ознак

In [ ]:
print(df["gender"].value_counts(dropna=False))

In [ ]:
print(df["smoking_history"].value_counts(dropna=False))

In [ ]:
display(df.head(10))

Вивести типи даних

In [ ]:
print(df.dtypes)

Дослідити цільову змінну: перевірити баланс класів

In [ ]:
print(df["diabetes"].value_counts())
print((df["diabetes"].value_counts(normalize=True)*100).round(2).astype(str)+" %")

**Висновок:** клас `diabetes = 1` трапляється істотно рідше за клас `0`; наявний дисбаланс. Точні частки наведені вище. Для навчання використаємо SMOTE тільки на train.

Перевірити наявність пропущених даних


In [ ]:
print(df.isna().sum())

Наявність дублікатів


In [ ]:
print("Кількість дублікатів:", df.duplicated().sum())

Видалити дублікати

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Після усунення дублікатів:", df.shape)

In [ ]:
display(df.head())

Ще раз перевірити баланс цільової змінної

In [ ]:
print(df["diabetes"].value_counts())
print((df["diabetes"].value_counts(normalize=True)*100).round(2))

Побудувати гістограму для цільової змінної


In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="diabetes", color="tab:blue")
plt.title("Розподіл цільової змінної")
plt.tight_layout()
plt.show()

Переглянути описові статистики **навчальної вибірки**

In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from imblearn.over_sampling import SMOTE

In [ ]:
X = df.drop(columns=["diabetes"])
y = df["diabetes"].astype(int)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train_raw.shape, "Test:", X_test_raw.shape)

Закодувати категоріальні ознаки **після поділу**. Створюємо стовпці за категоріями з train; у test залишаємо точно такий самий набір стовпців.

In [ ]:
categorical_cols = ["gender", "smoking_history"]
X_train = pd.get_dummies(X_train_raw, columns=categorical_cols, dtype=int)
X_test = pd.get_dummies(X_test_raw, columns=categorical_cols, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
assert X_train.columns.equals(X_test.columns)
print("Кількість ознак після кодування:", X_train.shape[1])

In [ ]:
display(X_train.describe().T)

Перевірити кореляцію ознак із цільовою змінною **лише на train**

In [ ]:
train_correlations = X_train.assign(diabetes=y_train).corr(numeric_only=True)["diabetes"].drop("diabetes")
print(train_correlations.sort_values(key=abs, ascending=False).round(3))

In [ ]:
feature_corr = X_train.corr(numeric_only=True)
plt.figure(figsize=(13, 10))
sns.heatmap(feature_corr, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Кореляції ознак — лише train")
plt.tight_layout()
plt.show()

**Висновок:** визначаємо найсильніші зв’язки виключно за train (таблиця й теплова карта вище). Найбільша прогностична пов’язаність очікується для HbA1c, глюкози крові та віку. Кореляція не доводить причинності; сильніші зв’язки між dummy-ознаками однієї категорії можуть виникати через взаємовиключність категорій. Без додаткової валідації ознаки не видаляємо.

Перевірити викиди **лише на навчальній вибірці**

In [ ]:
continuous_cols = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, continuous_cols):
    sns.boxplot(y=X_train[col], ax=ax, color="steelblue")
    ax.set_title(col)
plt.tight_layout()
plt.show()
q1 = X_train[continuous_cols].quantile(0.25)
q3 = X_train[continuous_cols].quantile(0.75)
iqr = q3 - q1
outlier_counts = ((X_train[continuous_cols] < q1-1.5*iqr) | (X_train[continuous_cols] > q3+1.5*iqr)).sum()
print("Кількість значень за межами IQR на train:\n", outlier_counts)

**Висновок:** викиди перевіряємо, але в цій лабораторній роботі автоматично **не видаляємо**. Екстремальні значення глюкози та HbA1c можуть бути клінічно важливими.

**Необов’язковий приклад.** Якщо потрібно видалити викиди методом IQR, межі обчислюємо **тільки на train**, видаляємо відповідні рядки з `X_train` і `y_train`, а `X_test` та `y_test` не змінюємо. У цій роботі приклад не запускаємо.

In [ ]:
# Приклад розрахунку меж (без видалення):
q1_bmi, q3_bmi = X_train["bmi"].quantile([0.25, 0.75])
iqr_bmi = q3_bmi - q1_bmi
print("bmi IQR boundaries:", q1_bmi - 1.5*iqr_bmi, q3_bmi + 1.5*iqr_bmi)
# X_train, y_train, X_test і y_test залишаються незмінними.

Перевірити асиметрію розподілів ознак **на train**

In [ ]:
print("Асиметрія розподілів на train:\n", X_train[continuous_cols].skew().round(3))

In [ ]:
print("Пропуски у train:", int(X_train.isna().sum().sum()))
print("Пропуски у test:", int(X_test.isna().sum().sum()))

**Проміжний висновок:** цільова змінна незбалансована; кореляції та IQR розраховані тільки на навчальній вибірці. Значення за межами IQR не завжди є помилками вимірювання — їх не видаляємо автоматично. Тестові приклади не змінюємо й не балансуємо.

Масштабуємо ознаки: `fit_transform` лише для train, `transform` — для test.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Масштабування завершено:", X_train_scaled.shape, X_test_scaled.shape)

Балансуємо лише навчальну вибірку за допомогою SMOTE

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)
print("До SMOTE:\n", y_train.value_counts().sort_index())
print("Після SMOTE:\n", pd.Series(y_train_bal).value_counts().sort_index())

Оголошуємо моделі

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1500, random_state=RANDOM_STATE),
    "RidgeClassifier": RidgeClassifier(random_state=RANDOM_STATE),
    "SGDClassifier": SGDClassifier(random_state=RANDOM_STATE, max_iter=1500),
    "SVC": SVC(random_state=RANDOM_STATE, cache_size=512),
}

Навчаємо базові моделі та порівнюємо їх на **валідаційній частині train**. Тестова вибірка залишається для остаточної оцінки.

In [ ]:
# Відокремлюємо validation від train до масштабування та SMOTE.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.20, random_state=RANDOM_STATE, stratify=y_train
)

val_scaler = StandardScaler()
X_fit_scaled = val_scaler.fit_transform(X_fit)
X_val_scaled = val_scaler.transform(X_val)

val_smote = SMOTE(random_state=RANDOM_STATE)
X_fit_bal, y_fit_bal = val_smote.fit_resample(X_fit_scaled, y_fit)

baseline_results = {}
for name, estimator in models.items():
    print("\n", "=" * 16, name, "=" * 16)
    estimator.fit(X_fit_bal, y_fit_bal)
    pred = estimator.predict(X_val_scaled)

    baseline_results[name] = {
        "F1 (class 1)": f1_score(y_val, pred, pos_label=1, zero_division=0),
        "Recall (class 1)": recall_score(y_val, pred, pos_label=1, zero_division=0),
    }

    print(classification_report(y_val, pred, labels=[0, 1], zero_division=0))
    plt.figure(figsize=(4, 3))
    sns.heatmap(
        confusion_matrix(y_val, pred, labels=[0, 1]),
        annot=True, fmt="d", cmap="Blues"
    )
    plt.xlabel("Прогноз")
    plt.ylabel("Факт")
    plt.title(f"{name} (validation)")
    plt.tight_layout()
    plt.show()

display(pd.DataFrame.from_dict(baseline_results, orient="index")
        .sort_values("F1 (class 1)", ascending=False))

Оптимізувати гіперпараметри


In [ ]:
param_grids = {
    "LogisticRegression": {"model__C": [0.1, 1.0], "model__solver": ["lbfgs"]},
    "RidgeClassifier": {"model__alpha": [0.1, 1.0, 10.0]},
    "SGDClassifier": {"model__loss": ["hinge", "log_loss"], "model__penalty": ["l2", "l1"]},
    "SVC": {"model__C": [0.1, 1.0], "model__kernel": ["linear", "rbf"]},
}

Підбираємо параметри методом **HalvingGridSearchCV** із `cv=3`, `scoring="f1"`. Масштабування та SMOTE виконуються в кожному навчальному згині, щоб не було витоку даних. На малих порціях для пошуку задаємо достатній мінімальний розмір навчальних даних.

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV
from imblearn.pipeline import Pipeline

In [ ]:
best_models, best_scores, tuning_results = {}, {}, {}
for name, estimator in models.items():
    print(f"\nПошук параметрів: {name}")
    pipeline = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", estimator),
    ])
    search = HalvingGridSearchCV(
        pipeline, param_grids[name], cv=3, scoring="f1", min_resources=300,
        factor=3, n_jobs=-1, refit=True, error_score="raise"
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    best_scores[name] = float(search.best_score_)
    predictions = search.predict(X_test)
    tuning_results[name] = {
        "CV F1": search.best_score_,
        "Test F1": f1_score(y_test, predictions, zero_division=0),
        "Test recall": recall_score(y_test, predictions, zero_division=0),
    }
    print("Найкращі параметри:", search.best_params_)
    print("Середній F1 (CV):", round(search.best_score_, 4))
    print(classification_report(y_test, predictions, zero_division=0))
    plt.figure(figsize=(4, 3))
    sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt="d", cmap="Blues")
    plt.title(f"{name}: після оптимізації")
    plt.xlabel("Прогноз")
    plt.ylabel("Факт")
    plt.tight_layout()
    plt.show()
display(pd.DataFrame(tuning_results).T.sort_values("CV F1", ascending=False))

Вибрати найкращу модель за середнім F1 на кросвалідації (а не за тестовою вибіркою)

In [ ]:
best_name = max(best_scores, key=best_scores.get)
best_model = best_models[best_name]
print("Найкраща модель за CV F1:", best_name)
print("Середній F1 за CV:", round(best_scores[best_name], 4))

In [ ]:
sample_indices = X_test.sample(n=10, random_state=RANDOM_STATE).index
sample_predictions = best_model.predict(X_test.loc[sample_indices])
comparison = pd.DataFrame({
    "індекс": sample_indices,
    "істинний клас": y_test.loc[sample_indices].to_numpy(),
    "передбачений клас": sample_predictions
})
display(comparison.reset_index(drop=True))

**Підсумкові висновки** (числові значення обчислюються під час запуску клітинок):

1. **Баланс класів:** позитивний клас значно рідший. Розподіл після видалення дублікатів і частки наведено вище. Кореляцію, асиметрію та IQR досліджено тільки на `train`; викиди не вилучено.
2. **Порівняння моделей:** таблиця `baseline_results` показує F1 і recall позитивного класу на validation. Після оптимізації метрики зведено в `tuning_results`. Модель обирається за **середнім F1 на CV**, а не за test.
3. **Параметри й помилки:** точні найкращі параметри надруковані після кожного пошуку. У матриці плутанини `FN` — фактичний діабет, який модель не розпізнала (нижня ліва клітинка), `FP` — хибне позитивне рішення (верхня права).
4. **Захист від перенавчання:** test відокремлений до статистичного аналізу й перетворень. Всередині CV `StandardScaler` і `SMOTE` підлаштовуються лише на тренувальній частині кожного згину. Це запобігає витоку даних і дає чеснішу оцінку узагальнення.

**Обмеження:** дані мають навчальне призначення; наведені результати не є медичним діагнозом. Під час послідовного порівняння багатьох моделей тест варто переглядати лише після остаточного вибору моделі.

**Додаткове завдання.**

Реалізувати моделі **XGBoost, LightGBM та CatBoost.**

Вивести класифікаційний звіт та матрицю плутанини для кожної моделі.

Оцінити, зробити висновки про отримані метрики

In [ ]:
# Додаткове завдання: три ансамблеві моделі.
# За необхідності у Colab: !pip -q install xgboost lightgbm catboost
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

extra_models = {
    "XGBoost": XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.08,
                              eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM": LGBMClassifier(n_estimators=150, max_depth=5, learning_rate=0.08,
                               verbose=-1, random_state=RANDOM_STATE, n_jobs=-1),
    "CatBoost": CatBoostClassifier(iterations=150, depth=5, learning_rate=0.08,
                                   verbose=False, random_seed=RANDOM_STATE, thread_count=-1),
}
extra_results = {}
for name, estimator in extra_models.items():
    # Для порівняння використовуємо лише train, SMOTE для ансамблів не обов'язковий.
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    extra_results[name] = {
        "F1 (class 1)": f1_score(y_test, pred, zero_division=0),
        "Recall (class 1)": recall_score(y_test, pred, zero_division=0),
    }
    print(f"\n{name}\n", classification_report(y_test, pred, zero_division=0))
    plt.figure(figsize=(4, 3))
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Greens")
    plt.title(name)
    plt.xlabel("Прогноз")
    plt.ylabel("Факт")
    plt.tight_layout()
    plt.show()
display(pd.DataFrame(extra_results).T.sort_values("F1 (class 1)", ascending=False))
print("Висновок: порівняйте F1 та recall класу 1 у таблиці вище."
      " Ансамблі не беруть участі у виборі best_model, який здійснювався окремо за CV.")